## 1. 字母异位词分组

- **难度**：中等
- **标签**：字符串 / 哈希表 / 数组计数

**题目**：给定一个字符串数组，将字母异位词组合在一起。字母异位词指字母相同但排列不同的字符串。可按任意顺序返回结果列表。

**示例**：
```python
输入：strs = ["eat", "tea", "tan", "ate", "nat", "bat"]
输出：[["bat"], ["nat", "tan"], ["ate", "eat", "tea"]]
解释：
  - "bat" 没有异位词
  - "nat" 和 "tan" 是异位词
  - "ate"、"eat"、"tea" 是异位词
```

**思路**：（计数法）字母异位词的本质是「各字母出现次数完全相同」，与顺序无关。

1. 用普通字典 `groups = {}` 做分组容器，key 为「字母计数指纹」，value 为同组的原始字符串列表。
2. 对每个字符串开一个长度 26 的计数数组 `count`，遍历字符：`count[ord(char) - ord('a')] += 1`。
3. 由于 `list` 不可哈希、不能直接当字典的 key，转成 `tuple(count)` 作为 key。
4. 若 `key` 尚未出现过，先 `groups[key] = []` 初始化该组，再 `groups[key].append(string)` 归组。
5. 时间复杂度：O(n·k)，n 为字符串数量、k 为最长字符串长度（每个字符串整体遍历一次）；空间复杂度：O(n·k)。

**亮点**：把「是否为字母异位词」这个等价关系编码成一个**可哈希的指纹**（26 元 tuple），用哈希表一次遍历即可完成分组。关键细节是 `list → tuple` 的转换——绕开「列表不可哈希」的限制；首次遇到新指纹时显式初始化空组，避免直接 append 触发 `KeyError`。相比排序法（O(n·k·log k)）无需排序、对长串更优。

**对比排序法**：对每个字符串排序后作 key（`key = ''.join(sorted(s))`），代码更短但单串需 O(k·log k)。两者核心思想一致——「为每个字符串生成一个等价类指纹」。

In [ ]:
from typing import List

class Solution:
    def groupAnagrams(self, strs: List[str]) -> List[List[str]]:
        # 异位词的特征就是：不考虑顺序，但是字母的num一致
        # s只有26个字母，用长度26的计数数组刻画字母构成
        groups = {} # 利用计数结果的唯一性转为tuple后做key， value就是原来的string list

        for string in strs:
            count = [0] * 26

            for ch in string:
                index = ord(ch) - ord("a")
                count[index] += 1
            
            key = tuple(count)

            if key not in groups:
                groups[key] = []
            
            groups[key].append(string)

        return list(groups.values())

# 测试
sol = Solution()

# 测试用例 1
strs1 = ["eat", "tea", "tan", "ate", "nat", "bat"]
result1 = sol.groupAnagrams(strs1)
print(f"测试 1: {result1}")  # 预期: 包含 ["bat"], ["nat", "tan"], ["ate", "eat", "tea"]

# 测试用例 2
strs2 = [""]
result2 = sol.groupAnagrams(strs2)
print(f"测试 2: {result2}")  # 预期: [[""]]

# 测试用例 3
strs3 = ["a"]
result3 = sol.groupAnagrams(strs3)
print(f"测试 3: {result3}")  # 预期: [["a"]]

## 2. 无重复字符的最长子串

- **难度**：中等
- **标签**：字符串 / 哈希表 / 滑动窗口

**题目**：给定一个字符串 `s`，找出其中不含有重复字符的 最长 子串的长度。

**示例**：
```python
输入：s = "abcabcbb"
输出：3
解释：因为无重复字符的最长子串是 "abc"，其长度为 3

输入：s = "bbbbb"
输出：1
解释：因为无重复字符的最长子串是 "b"，其长度为 1

输入：s = "pwwkew"
输出：3
解释：因为无重复字符的最长子串是 "wke"，其长度为 3
```

**思路**：滑动窗口 + 哈希表。维护一个窗口 `[left, right]`，其中没有重复字符。遍历字符串，当遇到重复字符时，移动 `left` 边界直到重复字符被移出窗口。用 `set` 记录窗口内的字符。时间 O(n)，空间 O(min(m, n))，m 为字符集大小。

**亮点**：滑动窗口的单向推进特性保证了 O(n) 时间复杂度；用 `set` 而非哈希表记录字符位置简化了逻辑，只需判断存在性而非精确位置。

In [ ]:
class Solution:
    def lengthOfLongestSubstring(self, s: str) -> int:
        # 使用集合维护当前滑动窗口中的字符
        # 窗口范围为 [left, right]，保证窗口内没有重复字符
        st = set()

        # left 指向窗口左边界
        left = 0

        # 记录无重复字符最长子串的长度
        ans = 0

        # right 作为右指针不断向右扩展窗口
        for right in range(len(s)):

            # 如果加入 s[right] 后会产生重复字符，
            # 则不断移动左指针，缩小窗口，
            # 直到窗口内不再包含 s[right]
            while s[right] in st:
                st.remove(s[left])
                left += 1

            # 将当前字符加入窗口
            # 此时窗口 [left, right] 内保证所有字符唯一
            st.add(s[right])

            # 更新当前最长无重复子串长度
            # 因为窗口是闭区间，所以长度为 right - left + 1
            ans = max(ans, right - left + 1)

        return ans

## 3. 最小覆盖子串

- **难度**：困难
- **标签**：字符串 / 滑动窗口 / 哈希表 / 双指针

**题目**：给定字符串 `s` 和 `t`，返回 `s` 中最短的、包含 `t` 中所有字符（含重复）的子串。若不存在则返回空字符串 `""`。保证答案唯一。

**示例**：
```python
输入：s = "ADOBECODEBANC", t = "ABC"
输出："BANC"
解释：最短覆盖子串是 "BANC"，包含 A、B、C 各一个

输入：s = "a", t = "a"
输出："a"

输入：s = "a", t = "aa"
输出：""
解释：s 中只有一个 'a'，无法覆盖两个
```

**思路**：滑动窗口 + 哈希表。
- `target` 统计 `t` 中每种字符的需求量；`cur_window` 统计当前窗口各字符数量；`valid` 记录窗口中「数量已达标」的字符种类数。
- 右指针 `right` 不断扩张，纳入新字符并更新 `valid`（仅当某字符数量**恰好**达到需求时 `valid += 1`，判等用 `==`，避免重复字符多次累加）。
- 当 `valid == len(target)` 时窗口覆盖 `t`，用 `while` 收缩左端点 `left` 寻找更短解：每次收缩前更新答案；移出字符后若其数量低于需求（`< target[remove]`）则 `valid -= 1`。
- 时间 O(m+n)（左右指针各最多遍历 `s` 一次），空间 O(字符集大小)。

**亮点**：用 `valid` 统计「达标的字符种类数」，把「窗口是否覆盖 t」的判断从 O(字符集) 降到 O(1)；经典的「右扩左缩」框架——右指针负责找可行解，左指针负责把可行解优化到最短。

In [ ]:
class Solution:
    def minWindow(self, s: str, t: str) -> str:
        # 双指针+哈希表字典，覆盖的条件是，当前窗口内的各种字符的个数>=t中的各种字符的数量 ：valid == len(target)
        target = {}
        for c in t:
            target[c] = target.get(c, 0) + 1
        
        cur_window = {}
        left = 0
        valid = 0
        ans = ""
        ans_len = float("inf")

        for right in range(len(s)):
            c = s[right]
            cur_window[c] = cur_window.get(c, 0) + 1

            if c in target and cur_window[c] == target[c]: #注意这里是==
                valid += 1
            
            # 当前窗口满足条件
            while valid == len(target):
                # 更新答案
                if right-left+1 < ans_len:
                    ans_len = right-left+1
                    ans = s[left:right+1] # 切片：左闭右开
                
                # 左边移出
                remove = s[left]
                cur_window[remove] -= 1
                if remove in target and cur_window[remove] < target[remove]:
                    valid -= 1
                
                left += 1
            
        return ans

# 测试
sol = Solution()
print(sol.minWindow("ADOBECODEBANC", "ABC"))  # "BANC"
print(sol.minWindow("a", "a"))                 # "a"
print(sol.minWindow("a", "aa"))                # ""

## 4. 找到字符串中所有字母异位词

- **难度**：中等
- **标签**：字符串 / 滑动窗口 / 定长窗口 / 数组计数

**题目**：给定字符串 `s` 和 `p`，找到 `s` 中所有 `p` 的异位词子串的起始索引（异位词：字母频率相同，顺序无关）。答案顺序不限。

**示例**：
```python
输入：s = "cbaebabacd", p = "abc"
输出：[0, 6]
解释：起始索引 0 的 "cba" 和索引 6 的 "bac" 都是 "abc" 的异位词

输入：s = "abab", p = "ab"
输出：[0, 1, 2]
```

**思路**：定长滑动窗口。窗口大小固定为 `len(p)`。
- `need[26]` 统计 `p` 的字符频率；`window[26]` 统计当前窗口频率。
- `right` 不断右移加入字符；一旦窗口长度超过 `len(p)`，立即移出左端字符、`left++`，保持窗口定长。
- 当 `window == need`（两个长度为 26 的数组逐元素相等）时，窗口即为 `p` 的异位词，记录起始索引 `left`。
- 时间 O(m)（每轮 O(26) 比较为常数），空间 O(1)（26 个字母）。

**亮点**：定长窗口让逻辑极简——窗口大小固定，只需「右进左出」平移，无需像最小覆盖子串那样判断何时收缩；用长度 26 的数组而非字典，可直接 `==` 比较频率，代码极简。

In [ ]:
from typing import List

class Solution:
    def findAnagrams(self, s: str, p: str) -> List[int]:
        # 滑动窗口+双指针
        res = []

        # p的字符频率
        need = [0] * 26
        for c in p:
            need[ord(c)-ord('a')] += 1
        
        window = [0] * 26
        left = 0

        for right in range(len(s)):
            # 加入window
            window[ord(s[right]) - ord('a')] += 1

            if right - left + 1 > len(p):
                window[ord(s[left]) - ord('a')] -= 1
                left += 1
            
            if window == need:
                res.append(left)
        return res

# 测试
sol = Solution()
print(sol.findAnagrams("cbaebabacd", "abc"))  # [0, 6]
print(sol.findAnagrams("abab", "ab"))          # [0, 1, 2]